### QASM 3.0 circuit
3 qubit repetition code

In [1]:
import pyqasm

qasm_code = """OPENQASM 3.0;
include "stdgates.inc";

qubit[5] q;
bit[2] syndrome;
const int max_iterations = 3;  // Valid constant declaration

// Encoding
reset q[0];
cx q[0], q[1];
cx q[0], q[2];

// Stabilization loop (verified syntax)
for const int i in [0:max_iterations] {  // OFFICIAL SYNTAX: [start:end]
    reset q[3];
    reset q[4];
    
    // First stabilizer
    cx q[0], q[3];
    cx q[1], q[3];
    measure q[3] -> syndrome[0];
    
    // Second stabilizer
    cx q[1], q[4];
    cx q[2], q[4];
    measure q[4] -> syndrome[1];
    
    // Correction logic
    if (syndrome == 1) {
        x q[2];
    } else if (syndrome == 2) {
        x q[0];
    } else if (syndrome == 3) {
        x q[1];
    }
}

// Final measurement
bit[3] result;
measure q[0] -> result[0];
measure q[1] -> result[1];
measure q[2] -> result[2];"""


module = pyqasm.loads(qasm_code)
module.unroll()

# print(pyqasm.dumps(module))

line 14:4 no viable alternative at input 'const'


ValidationError: Failed to parse OpenQASM string: 

In [2]:
import ucc

# qBraid conversion fails due to classical register lacking support in PyQASM (inside qBraid translate step)
ucc.compile(qasm_code)

line 14:4 no viable alternative at input 'const'


ProgramConversionError: Failed to convert 'qasm3' to 'qiskit' due to the following error(s):

Conversion qasm3 -> qiskit failed due to exception raised while converting from 'qasm3'.
QASM3ParsingError: 


### Official Qiskit example for QASM 3.0
Try an example straight from Qiskit (not QEC, but contains classical operations)

In [4]:
qasm_code_qis_ex = """
    OPENQASM 3.0;
    include "stdgates.inc";
 
    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;
 
    let aliased = q[0:1];
 
    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }
 
    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];
 
    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }
 
    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }
 
    out = measure q;
"""

In [5]:
import ucc

# qBraid conversion fails due to classical register lacking support in PyQASM (inside qBraid translate step)
ucc.compile(qasm_code_qis_ex)

TranspilerError: 'Unable to translate the operations in the circuit: ["my_gate", "measure", "while_loop", "reset", "my_phase", "if_else"] to the backend\'s (or manually specified) target basis: {"cx", "rz", "ry", "rx", "h", "measure", "reset", "barrier", "snapshot", "delay", "store"}. This likely means the target basis is not universal or there are additional equivalence rules needed in the EquivalenceLibrary being used. For more details on this error see: https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.passes. BasisTranslator#translation-errors'

### Qiskit circuit
3 qubit repetition code

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 1)
qc.x([0, 2])
qc.measure_all()  # Measure all qubits.